In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R_
from pathlib import Path
from pid_optimizer.plant_model import PlantModel, Inertia, Drag
from pid_optimizer.gains import HERMIT_REFERENCE_GAINS
from pid_optimizer.pid_simulator import PX4Simulator, compute_rmse
from pid_optimizer.ga import GeneticAlgorithm

PLANT_PATH   = "../artifacts/plant_model.json"
CHECKPOINT   = "../artifacts/ga_plant_checkpoint.json"
BEST_PLANT   = "../artifacts/best_plant.json"
LOG_PATH     = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"

plant = PlantModel.load(PLANT_PATH)
print(f"Loaded plant: {plant}")


Loaded plant: PlantModel(mass_kg=1.8, kT=23.807374954223633, tau_motor_s=0.005003370182525293, inertia=Inertia(Ixx=0.01728191388581314, Iyy=0.009356793261798222, Izz=0.021638707147611364), drag=Drag(kD_xy=0.15, kD_z=0.2), arm_length_m=0.17, source_log='../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg', fit_rmse={})


In [2]:

log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0  = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

DT, T_MAX = 0.004, 20.0

pos_sp_df = to_df(log, "vehicle_local_position_setpoint")
pos_df    = to_df(log, "vehicle_local_position")
att_df    = to_df(log, "vehicle_attitude")
thrust_df = to_df(log, "vehicle_thrust_setpoint")

t_start_raw = max(pos_df["t_s"].min(), att_df["t_s"].min(), thrust_df["t_s"].min())
t_end       = min(thrust_df["t_s"].max(), att_df["t_s"].max(), T_MAX)
t_sim       = np.arange(t_start_raw, t_end, DT)

def interp_col(src_df, col, t_out):
    mask = ~src_df[col].isna()
    if mask.sum() < 2:
        return np.zeros(len(t_out))
    return np.interp(t_out, src_df["t_s"][mask].values, src_df[col][mask].values)

# Real attitude interpolated onto t_sim
q_arr     = att_df[["q[0]","q[1]","q[2]","q[3]"]].values
euler_arr = R_.from_quat(q_arr[:, [1,2,3,0]]).as_euler("xyz")
t_att     = att_df["t_s"].values
euler_sim = pd.DataFrame({
    "roll":  np.interp(t_sim, t_att, euler_arr[:, 0]),
    "pitch": np.interp(t_sim, t_att, euler_arr[:, 1]),
    "yaw":   np.interp(t_sim, t_att, euler_arr[:, 2]),
})

# Thrust: normalized throttle -> physical force [N]
# Calibrate kT from hover (stable altitude, z < -0.5m, |vz| < 0.1 m/s)
def _thr_to_F(thr_arr, kT):
    return kT * 4.0 * thr_arr**2

_z_cal   = np.interp(t_sim, pos_df["t_s"].values, pos_df["z"].values)
_vz_col  = "vz" if "vz" in pos_df.columns else None
_vz_cal  = (np.interp(t_sim, pos_df["t_s"].values, pos_df["vz"].values)
            if _vz_col else np.gradient(_z_cal, DT))
_thr_cal = np.abs(np.interp(t_sim, thrust_df["t_s"].values, thrust_df["xyz[2]"].values))
hover_mask = (_z_cal < -0.5) & (np.abs(_vz_cal) < 0.1) & (_thr_cal > 0.1)

if hover_mask.sum() > 50:
    thr_hover      = np.mean(_thr_cal[hover_mask])
    kT_calibrated  = (plant.mass_kg * 9.81) / (4.0 * thr_hover**2)
    print(f"Hover calibration: thr_hover={thr_hover:.4f}, kT_calibrated={kT_calibrated:.2f}")
else:
    kT_calibrated = plant.kT
    print(f"Hover phase not found, using plant.kT={plant.kT:.2f}")

thr_raw    = np.abs(np.interp(t_sim, thrust_df["t_s"].values, thrust_df["xyz[2]"].values))
thrust_sim = pd.DataFrame({"F": _thr_to_F(thr_raw, kT=kT_calibrated)})

# Reference position (real EKF) for fitness evaluation
actual_pos = pd.DataFrame({
    "x": interp_col(pos_df, "x", t_sim),
    "y": interp_col(pos_df, "y", t_sim),
    "z": interp_col(pos_df, "z", t_sim),
})

# Initial state from ULG at t_start
def _idx_at(df, t):
    return min(np.searchsorted(df["t_s"].values, t), len(df) - 1)

x0_pos  = pos_df[["x","y","z"]].iloc[_idx_at(pos_df, t_sim[0])].values
vel_cols = [c for c in ["vx","vy","vz"] if c in pos_df.columns]
x0_vel  = pos_df[vel_cols].iloc[_idx_at(pos_df, t_sim[0])].values if vel_cols else np.zeros(3)
x0      = np.concatenate([x0_pos, x0_vel])
print(f"Initial pos={x0_pos.round(3)}, vel={x0_vel.round(3)}")

# Baseline RMSE with current plant
ref_result = PX4Simulator(plant, HERMIT_REFERENCE_GAINS).run_translational(
    thrust_sim, euler_sim, dt=DT, x0=x0)
baseline_rmse = compute_rmse(ref_result[["x","y","z"]], actual_pos, cols=["x","y","z"])
print("Baseline RMSE:", {k: f"{v:.4f}" for k, v in baseline_rmse.items()})


Hover calibration: thr_hover=0.4406, kT_calibrated=22.74
Initial pos=[ 0.018  0.038 -0.055], vel=[0.001 0.    0.009]
Baseline RMSE: {'x': '0.4258', 'y': '1.9827', 'z': '0.7437'}


In [3]:
# Plant parameter bounds: [mass, kT, kD_xy, kD_z]
PLANT_BOUNDS = [(1.0, 3.0), (10.0, 50.0), (0.0, 0.5), (0.0, 0.8)]
WEIGHTS = {"x": 1.5, "y": 1.5, "z": 2.0}

def fitness(params: np.ndarray) -> float:
    mass, kT, kD_xy, kD_z = params
    try:
        plant_trial = PlantModel(
            mass_kg=mass, kT=kT, tau_motor_s=plant.tau_motor_s,
            inertia=plant.inertia,
            drag=Drag(kD_xy=kD_xy, kD_z=kD_z),
            arm_length_m=plant.arm_length_m)
        F_trial = kT * 4.0 * thr_raw**2
        thrust_trial = pd.DataFrame({"F": F_trial})
        result = PX4Simulator(plant_trial, HERMIT_REFERENCE_GAINS).run_translational(
            thrust_trial, euler_sim, dt=DT, x0=x0)
        rmse = compute_rmse(result[["x","y","z"]], actual_pos, cols=["x","y","z"])
        score = sum(WEIGHTS[k] * rmse[k] for k in WEIGHTS)
        return float(score) if np.isfinite(score) else 1e6
    except Exception:
        return 1e6

ref_score = fitness([plant.mass_kg, kT_calibrated, plant.drag.kD_xy, plant.drag.kD_z])
print(f"Reference plant fitness: {ref_score:.4f}")

Reference plant fitness: 5.1002


In [4]:
import os, json

GENERATIONS = 5  # Set to 200 for real optimization
CHECKPOINT_EVERY = 5
BEST_PLANT = "../artifacts/best_plant.json"

if os.path.exists(CHECKPOINT):
    ga = GeneticAlgorithm(PLANT_BOUNDS, fitness_fn=fitness, pop_size=20, seed=42)
    ga.load_checkpoint(CHECKPOINT)
    print(f"Resumed from generation {ga.generation}, best fitness={ga.best_fitness:.4f}")
else:
    ga = GeneticAlgorithm(PLANT_BOUNDS, fitness_fn=fitness, pop_size=20, seed=42)
    ga.initialize()
    print(f"Initialized. Generation 0 best fitness={ga.best_fitness:.4f}")

remaining = GENERATIONS - ga.generation
print(f"Running {remaining} more generations...")
for gen in range(remaining):
    ga.step()
    if ga.generation % CHECKPOINT_EVERY == 0:
        ga.save_checkpoint(CHECKPOINT)
        print(f"Gen {ga.generation:4d} | best={ga.best_fitness:.4f} | mean={ga.history[-1]['mean']:.4f}")

best_idx = ga.scores.argmin()
best_params = ga.population[best_idx]
mass_b, kT_b, kD_xy_b, kD_z_b = best_params

best_plant_data = {
    "reference_fitness": ref_score,
    "best_fitness": float(ga.best_fitness),
    "params": {"mass_kg": float(mass_b), "kT": float(kT_b),
               "kD_xy": float(kD_xy_b), "kD_z": float(kD_z_b)},
}
with open(BEST_PLANT, "w") as f:
    json.dump(best_plant_data, f, indent=2)

print(f"\nBest fitness: {ga.best_fitness:.4f}  (reference: {ref_score:.4f})")
print(f"Best params: mass={mass_b:.3f} kg, kT={kT_b:.2f}, kD_xy={kD_xy_b:.4f}, kD_z={kD_z_b:.4f}")
print(f"Saved to {BEST_PLANT}")

Resumed from generation 5, best fitness=3.1259
Running 0 more generations...

Best fitness: 3.1259  (reference: 5.1002)
Best params: mass=2.848 kg, kT=10.00, kD_xy=0.4061, kD_z=0.6246
Saved to ../artifacts/best_plant.json


In [5]:
if ga.history:
    gens  = [h["generation"] for h in ga.history]
    bests = [h["best"]  for h in ga.history]
    means = [h["mean"]  for h in ga.history]
    stds  = [h["std"]   for h in ga.history]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(gens, bests, label="best", linewidth=2)
    ax.plot(gens, means, label="mean", alpha=0.7)
    ax.fill_between(gens, np.array(means)-np.array(stds),
                    np.array(means)+np.array(stds), alpha=0.2, label="±1σ")
    ax.axhline(ref_score, color="red", linestyle="--", label="reference plant")
    ax.set_xlabel("Generation"); ax.set_ylabel("Fitness (weighted RMSE [m])")
    ax.set_title("GA Plant Identification Convergence"); ax.legend()
    plt.tight_layout(); plt.savefig("/tmp/ga_convergence.png"); plt.close()
    print("Saved convergence plot to /tmp/ga_convergence.png")
else:
    print("No history to plot")

Saved convergence plot to /tmp/ga_convergence.png


In [6]:
param_names = ["mass_kg", "kT", "kD_xy", "kD_z"]
lo = np.array([b[0] for b in PLANT_BOUNDS])
hi = np.array([b[1] for b in PLANT_BOUNDS])
top10 = ga.population[ga.scores.argsort()[:min(10, len(ga.population))]]
normalized = (top10 - lo) / (hi - lo + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of top individuals
ax = axes[0]
im = ax.imshow(normalized.T, aspect="auto", cmap="coolwarm", vmin=0, vmax=1)
ax.set_yticks(range(len(param_names))); ax.set_yticklabels(param_names, fontsize=9)
ax.set_xlabel("Top individuals"); ax.set_title("Plant params (normalized to bounds)")
plt.colorbar(im, ax=ax, label="0=low bound, 1=high bound")

# Best-plant 3-curve position comparison
best_plant_trial = PlantModel(
    mass_kg=float(mass_b), kT=float(kT_b), tau_motor_s=plant.tau_motor_s,
    inertia=plant.inertia, drag=Drag(kD_xy=float(kD_xy_b), kD_z=float(kD_z_b)),
    arm_length_m=plant.arm_length_m)
F_best = float(kT_b) * 4.0 * thr_raw**2
thrust_best = pd.DataFrame({"F": F_best})
result_best = PX4Simulator(best_plant_trial, HERMIT_REFERENCE_GAINS).run_translational(
    thrust_best, euler_sim, dt=DT, x0=x0)
rmse_best = compute_rmse(result_best[["x","y","z"]], actual_pos, cols=["x","y","z"])

t_plot = t_sim[:len(result_best)]
ax2 = axes[1]
for col, color in [("z", "tab:blue")]:
    ax2.plot(t_plot, result_best[col].values, color=color, label=f"sim {col}", linewidth=1.5)
    ax2.plot(t_sim, actual_pos[col].values, color="tab:orange", label=f"real {col}", alpha=0.8)
ax2.set_xlabel("Time (s)"); ax2.set_ylabel("Z [m NED]")
ax2.set_title(f"Best plant Z  (RMSE x={rmse_best['x']:.2f} y={rmse_best['y']:.2f} z={rmse_best['z']:.2f} m)")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig("/tmp/ga_heatmap.png"); plt.close()
print("Saved heatmap to /tmp/ga_heatmap.png")
print(f"Best RMSE: x={rmse_best['x']:.3f} m, y={rmse_best['y']:.3f} m, z={rmse_best['z']:.3f} m")

Saved heatmap to /tmp/ga_heatmap.png
Best RMSE: x=0.580 m, y=0.357 m, z=0.860 m
